In [1]:
import os
import sys
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

from utils.summary import get_model_stats

import torch
import torch.nn.functional as F

In [1]:
Ts = [1, 5, 20, 40]
Ts.index(20)

2

In [2]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
from data.voc import get_voc_pipeline

train_loader, val_loader, test_loader = get_voc_pipeline(batch_size=4)
sample_x, sample_y = next(iter(train_loader))
print(sample_x.shape)
print(sample_y.shape)

Using downloaded and verified file: /home/dodogama/code/project_compression/data/VOCtrainval_11-May-2012.tar
Extracting /home/dodogama/code/project_compression/data/VOCtrainval_11-May-2012.tar to /home/dodogama/code/project_compression/data
Using downloaded and verified file: /home/dodogama/code/project_compression/data/VOCtrainval_11-May-2012.tar
Extracting /home/dodogama/code/project_compression/data/VOCtrainval_11-May-2012.tar to /home/dodogama/code/project_compression/data
torch.Size([4, 3, 256, 256])
torch.Size([4, 1, 256, 256])


# Hmmm

In [ ]:
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from PIL import Image
import requests

feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = feature_extractor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits  # shape (batch_size, num_labels, height/4, width/4)

input_tensor = torch.randn(1, 3, 256, 256)  # Example input
model(input_tensor).logits.shape

logits.shape
inputs.keys()
get_model_stats(model, )

# Stuff

In [ ]:
class ConstantWithWarmup(torch.optim.lr_scheduler._LRScheduler):
    def __init__(
        self,
        optimizer,
        num_warmup_steps: int,
    ):
        self.num_warmup_steps = num_warmup_steps
        super().__init__(optimizer)

    def get_lr(self):
        if self._step_count <= self.num_warmup_steps:
            # warmup
            scale = 1.0 - (self.num_warmup_steps - self._step_count) / self.num_warmup_steps
            lr = [base_lr * scale for base_lr in self.base_lrs]
            self.last_lr = lr
        else:
            lr = self.base_lrs
        return lr
lr_scheduler = ConstantWithWarmup(optimizer, WARMUP_STEPS)

    
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # Linear warmup
    return 1.0

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

In [ ]:
import torch
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR, MultiStepLR, SequentialLR

optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9)

# Define the warmup scheduler
warmup_steps = 500
warmup_scheduler = LambdaLR(optimizer, lr_lambda=lambda step: min(1.0, step / warmup_steps))

# Define the decay scheduler (e.g., MultiStepLR for step decay)
decay_epochs = [30, 60, 90]  # Epoch milestones for decay
gamma = 0.1  # Decay factor
decay_scheduler = MultiStepLR(optimizer, milestones=decay_epochs, gamma=gamma)

# Combine the schedulers using SequentialLR
schedulers = [warmup_scheduler, decay_scheduler]
milestones = [warmup_steps]  # Switch to decay after warmup_steps
scheduler = SequentialLR(optimizer, schedulers, milestones=milestones)

In [9]:
# Optional: Warm-up scheduler (PyTorch >= 1.10)
from torch.optim.lr_scheduler import SequentialLR, LinearLR

warmup = LinearLR(optimizer, start_factor=1e-3, total_iters=5)
main_sched = StepLR(optimizer, step_size=30, gamma=0.1)
scheduler = SequentialLR(optimizer, schedulers=[warmup, main_sched], milestones=[5])

In [ ]:
    student = mlp.mnist800().to(device)
    criterion = DistillationLoss(T=T)
    crossentropy = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=0.005)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    aux_metrics = {'accuracy': Accuracy()}
    path = f'../models/weights/student{T}.pth'